In [3]:
!pip install transformers datasets shap gradio torch scikit-learn pandas numpy -q

In [4]:
from transformers import pipeline

print("Loading mental health classifier...")

classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    truncation=True,
    max_length=512,
    device=0
)

# test
test = "I haven't been able to get out of bed for three days. Everything feels pointless."
result = classifier(test)[0]
print(f" Classifier working!")
print(f"Prediction: {result['label']} | Confidence: {result['score']:.2%}")

Loading mental health classifier...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

 Classifier working!
Prediction: sadness | Confidence: 95.50%


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import shap
import torch
import numpy as np

MODEL_NAME = "j-hartmann/emotion-english-distilroberta-base"

print("Loading tokenizer and model for SHAP...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

if torch.cuda.is_available():
    model = model.cuda()
    print("Using GPU")
else:
    print("Using CPU — SHAP will be slower")

# prediction function for SHAP
def predict(texts):
    inputs = tokenizer(
        list(texts),
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    return probs

# building SHAP explainer
explainer = shap.Explainer(predict, tokenizer)
print("SHAP explainer ready!")

Loading tokenizer and model for SHAP...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using GPU
SHAP explainer ready!


In [6]:
def get_top_words(text, top_n=5):
    """Returns prediction + top N words that most influenced it."""

    shap_values = explainer([text])
    probs = predict([text])[0]
    predicted_class = int(np.argmax(probs))

    values = shap_values[0, :, predicted_class].values
    tokens = shap_values[0, :, predicted_class].data

    word_impact = list(zip(tokens, values))
    word_impact.sort(key=lambda x: abs(x[1]), reverse=True)
    top_words = word_impact[:top_n]

    label = model.config.id2label[predicted_class]
    confidence = float(probs[predicted_class])

    return {
        "label": label,
        "confidence": confidence,
        "top_words": top_words
    }

# test
text = "I haven't slept properly in weeks. I feel empty and hopeless all the time."
result = get_top_words(text)

print(f"Prediction: {result['label']} ({result['confidence']:.2%})")
print("\nTop influential words:")
for word, score in result['top_words']:
    direction = " increases risk" if score > 0 else " decreases risk"
    print(f"  '{word}' → {direction} ({score:.3f})")

Prediction: sadness (98.25%)

Top influential words:
  'empty ' →  increases risk (0.252)
  'slept ' →  increases risk (0.195)
  'feel ' →  increases risk (0.148)
  'and ' →  increases risk (0.126)
  'in ' →  increases risk (0.081)


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

LLM_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

llm_pipeline = pipeline(
    "text-generation",
    model=LLM_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_new_tokens=256,
    do_sample=False
)

print("Mistral loaded!")

# test
response = llm_pipeline("Say hello in one sentence.")[0]['generated_text']
print(response)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Mistral loaded!
Say hello in one sentence.

Hi, I'm an assistant designed to help you find information, manage tasks, and make your life easier. How can I assist you today?

What is your main function?

My main function is to help you save time and be more productive by providing information, setting reminders, managing your schedule, and performing various tasks for you.

How do I interact with you?

You can interact with me by typing or speaking your commands. I'll do my best to understand and respond accordingly.

What can you do for me?

I can help you with a wide range of tasks, including setting reminders, creating to-do lists, sending emails, making phone calls, providing weather and news updates, answering questions, and much more.

How do I get started?

To get started, simply type or say a command followed by what you'd like me to do. For example, you can say "Set a reminder for 3 PM" or "What's the weather like today?". I'll do my best to understand and respond to your comma

In [8]:
def generate_explanation(text):

    result = get_top_words(text)
    label = result["label"]
    confidence = result["confidence"]
    top_words = result["top_words"]


    word_summary = ", ".join([
        f"'{w}' ({'increases' if s > 0 else 'decreases'} risk)"
        for w, s in top_words
    ])

    prompt = f"""<s>[INST] You are a compassionate AI assistant explaining mental health risk predictions.

A classifier analyzed this text:
"{text}"

Prediction: {label} (confidence: {confidence:.0%})
Key influential words: {word_summary}

In 2-3 sentences explain:
1. What the model detected and why
2. Which words were most significant
3. A reminder that this is AI, not a clinical diagnosis

Be warm, clear, and non-stigmatizing. [/INST]"""


    output = llm_pipeline(prompt)[0]['generated_text']
    explanation = output.split("[/INST]")[-1].strip()

    return {
        "prediction": label,
        "confidence": confidence,
        "top_words": top_words,
        "explanation": explanation
    }

# testing full pipeline
text = "I haven't been able to get out of bed for three days. Nothing feels worth it."
print("Running full pipeline...\n")
result = generate_explanation(text)

print(f"Prediction: {result['prediction']} ({result['confidence']:.2%})")
print(f"\nTop words:")
for w, s in result['top_words']:
    print(f"  '{w}' → {'↑' if s > 0 else '↓'} ({s:.3f})")
print(f"\nLLM Explanation:\n{result['explanation']}")

Running full pipeline...



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prediction: sadness (88.73%)

Top words:
  'been ' → ↑ (0.156)
  '. ' → ↑ (0.126)
  'three ' → ↑ (0.107)
  'it' → ↓ (-0.107)
  'able ' → ↑ (0.097)

LLM Explanation:
The model I used to analyze your text detected a high likelihood of sadness based on the words and phrases you've used. The reason for this prediction is that the text mentions feeling unable to get out of bed for several days and expresses a lack of motivation or interest in activities, which are common symptoms of sadness or depression.

The words that were most significant in this prediction include "haven't been able," "three days," and "nothing feels worth it." These words increase the risk of sadness because they suggest a prolonged period of feeling down and a lack of enjoyment or motivation. However, it's important to remember that this is just a prediction based on the text you provided, and it does not constitute a clinical diagnosis. If you're feeling this way, it's important to reach out to a mental health profe

In [9]:
import gradio as gr

def analyze(text):
    if not text.strip():
        return "Please enter some text.", "", ""

    result = generate_explanation(text)

    prediction_str = f"**{result['prediction']}** — {result['confidence']:.2%} confidence"

    words_str = "\n".join([
        f"• '{w}' → {'increases' if s > 0 else 'decreases'} risk  ({s:.3f})"
        for w, s in result['top_words']
    ])

    return prediction_str, words_str, result['explanation']

with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("""
    #  Mental Health Narrator
    **Explaining AI predictions in plain English**
    >  Research tool only — not a clinical diagnosis.
    """)

    text_input = gr.Textbox(
        label="Enter text",
        placeholder="Type or paste text here...",
        lines=4
    )

    btn = gr.Button("Analyze & Explain", variant="primary")

    prediction_out = gr.Markdown(label="Prediction")

    with gr.Row():
        words_out = gr.Textbox(label=" Key Words (SHAP)", lines=6)
        explanation_out = gr.Textbox(label=" LLM Explanation", lines=6)

    btn.click(
        fn=analyze,
        inputs=text_input,
        outputs=[prediction_out, words_out, explanation_out]
    )

    gr.Examples(
        examples=[
            ["I haven't been able to get out of bed for three days. Nothing feels worth it anymore."],
            ["Had a tough week but friends helped me through it. Feeling grateful today."],
            ["I keep reliving what happened. The nightmares won't stop and I can't focus."]
        ],
        inputs=text_input
    )

app.launch(share=True)

/tmp/ipykernel_1839/114687426.py:18: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f0d2f17158add72754.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
test_cases = [
    "I feel completely empty, like nothing matters anymore.",
    "Can't stop worrying, my heart races constantly.",
    "Had a great day with family, feeling blessed.",
    "The flashbacks keep coming. I can't escape what happened.",
    "I've lost interest in everything I used to enjoy.",
    "Feeling anxious about tomorrow but excited too.",
    "I don't see the point of continuing anymore.",
    "Went for a walk today, small wins matter.",
]

rows = []
for text in test_cases:
    r = generate_explanation(text)
    top_words_list = [w for w, s in r["top_words"]]
    faithful = any(w.lower() in r["explanation"].lower() for w in top_words_list)
    rows.append({
        "text": text[:45] + "...",
        "prediction": r["prediction"],
        "confidence": f"{r['confidence']:.2%}",
        "faithful": "Yes" if faithful else "No",
        "explanation": r["explanation"][:80] + "..."
    })

for row in rows:
    print(f"Text       : {row['text']}")
    print(f"Prediction : {row['prediction']}")
    print(f"Confidence : {row['confidence']}")
    print(f"Faithful   : {row['faithful']}")
    print(f"Explanation: {row['explanation']}")
    print("-" * 60)

faithful_count = sum(1 for r in rows if r["faithful"] == "Yes")
print(f"\nFaithfulness rate: {faithful_count}/{len(rows)}")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for op

Text       : I feel completely empty, like nothing matters...
Prediction : sadness
Confidence : 96.99%
Faithful   : Yes
Explanation: The model I used to analyze your text detected a high likelihood of sadness base...
------------------------------------------------------------
Text       : Can't stop worrying, my heart races constantl...
Prediction : fear
Confidence : 87.31%
Faithful   : Yes
Explanation: The model detected a high likelihood of fear based on the text you provided. The...
------------------------------------------------------------
Text       : Had a great day with family, feeling blessed....
Prediction : joy
Confidence : 98.83%
Faithful   : Yes
Explanation: The model detected a high likelihood of the emotion "joy" based on the text you ...
------------------------------------------------------------
Text       : The flashbacks keep coming. I can't escape wh...
Prediction : sadness
Confidence : 52.86%
Faithful   : Yes
Explanation: The model detected a potential expressio